# 04 — Regression Testing

## Why this notebook exists

In **notebook 03** we built a reusable eval harness: `run_eval(agent, dataset, graders)` runs a list of graders against every example in a dataset and hands back an `EvalReport` with pass rates and mean scores. That harness is the tool we needed — but using it once tells you how your agent performs *today*. It doesn't tell you whether it got *worse* between yesterday and today.

This notebook adds the missing layer: **regression testing**. We version an eval dataset (treat it as a checked-in artifact that evolves deliberately, not accidentally), run the harness against two different agent versions, compute a per-example delta table, and encode a **gate** that returns PASS or FAIL automatically. This is the deterministic precursor to the CI gate assembled in full in notebook 08 — the same logic, but wired by hand here so the mechanics are transparent.

No API key is needed. Both agent versions are deterministic Python stubs.

## What you'll learn

- How to treat an eval **dataset as a versioned artifact** — a list of `Example` objects you'd commit to source control so regressions are caught on the diff, not in production.
- How to compare two `EvalReport` objects with **`compare_reports`** — a per-example delta table showing which examples regressed, held steady, or improved.
- How to encode a **`regression_gate`** — a single boolean function that returns `True` (PASS) or `False` (FAIL) based on an explicit, auditable rule combining aggregate score floor and per-example regression tolerance.
- Why **determinism in graders** is a prerequisite for gates to be meaningful — a gate built on a flaky grader is a false sense of safety.
- How this deterministic gate is the foundation for the CI-style gate in notebook 08.

## 1. Setup — Re-Declare the Harness

This cell copies the shared primitives forward from notebook 03 verbatim so this notebook is fully self-contained. In a real project you'd package these into a module (e.g. `evals/harness.py`) and import from there; here we keep them inline for readability.

**Primitives declared here (identical signatures to nb 03):**
- `Score(key, score, passed, comment="")` — one grader's verdict on one example.
- `Example(input, expected=None, metadata={})` — a single eval case.
- `ExampleResult(example, output, scores)` — the output + all scores for one example.
- `EvalReport(results)` — aggregates a list of `ExampleResult`; exposes `.pass_rate(key=None)`, `.mean_score(key=None)`, `.summary_table()`.
- `run_eval(agent, dataset, graders) -> EvalReport` — runs every grader on every example.
- Graders: `exact_match`, `make_contains`, `make_regex`, `make_schema_grader`, `make_golden_grader`.

In [ ]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from typing import Any, Callable

from pydantic import BaseModel, ValidationError


# ---------------------------------------------------------------------------
# Core data types
# ---------------------------------------------------------------------------

@dataclass
class Score:
    """One grader's verdict on one example."""
    key: str
    score: float          # in [0.0, 1.0]
    passed: bool
    comment: str = ""


@dataclass
class Example:
    """A single evaluation case."""
    input: Any
    expected: Any = None
    metadata: dict = field(default_factory=dict)


@dataclass
class ExampleResult:
    """The output and all scores for one example."""
    example: Example
    output: Any
    scores: list[Score]

    def mean_score(self) -> float:
        """Average score across all graders for this example."""
        if not self.scores:
            return 0.0
        return sum(s.score for s in self.scores) / len(self.scores)


class EvalReport:
    """Aggregates ExampleResults into metrics."""

    def __init__(self, results: list[ExampleResult]) -> None:
        self.results = results

    def pass_rate(self, key: str | None = None) -> float:
        """Fraction of examples where all graders (or the named grader) passed."""
        if not self.results:
            return 0.0
        if key is None:
            passed = sum(
                1 for r in self.results if all(s.passed for s in r.scores)
            )
        else:
            passed = sum(
                1 for r in self.results
                for s in r.scores
                if s.key == key and s.passed
            )
        return passed / len(self.results)

    def mean_score(self, key: str | None = None) -> float:
        """Mean score across all examples (and all graders, or the named grader)."""
        if not self.results:
            return 0.0
        if key is None:
            scores = [s.score for r in self.results for s in r.scores]
        else:
            scores = [
                s.score
                for r in self.results
                for s in r.scores
                if s.key == key
            ]
        return sum(scores) / len(scores) if scores else 0.0

    def summary_table(self) -> str:
        """A readable per-example table with scores and pass/fail."""
        lines = [
            f"{'#':<4} {'Input':<35} {'Mean Score':<12} {'Passed?':<10} Scores",
            "-" * 80,
        ]
        for i, r in enumerate(self.results):
            input_str = str(r.example.input)[:33]
            mean = r.mean_score()
            all_passed = all(s.passed for s in r.scores)
            score_detail = ", ".join(
                f"{s.key}={s.score:.2f}({'P' if s.passed else 'F'})"
                for s in r.scores
            )
            lines.append(
                f"{i:<4} {input_str:<35} {mean:<12.3f} {'PASS' if all_passed else 'FAIL':<10} {score_detail}"
            )
        lines.append("-" * 80)
        lines.append(
            f"     pass_rate={self.pass_rate():.3f}  mean_score={self.mean_score():.3f}"
        )
        return "\n".join(lines)


# ---------------------------------------------------------------------------
# Eval runner
# ---------------------------------------------------------------------------

def run_eval(
    agent: Callable[[Any], Any],
    dataset: list[Example],
    graders: list[Callable[[Example, Any], Score]],
) -> EvalReport:
    """Run every grader on every example and return an EvalReport."""
    results: list[ExampleResult] = []
    for example in dataset:
        output = agent(example.input)
        scores = [grader(example, output) for grader in graders]
        results.append(ExampleResult(example=example, output=output, scores=scores))
    return EvalReport(results)


# ---------------------------------------------------------------------------
# Deterministic graders (attribute form: graders read example.expected)
# ---------------------------------------------------------------------------

def exact_match(example: Example, output: Any) -> Score:
    """Pass iff output == example.expected (string comparison after strip)."""
    expected = str(example.expected).strip()
    actual = str(output).strip()
    passed = actual == expected
    return Score(
        key="exact_match",
        score=1.0 if passed else 0.0,
        passed=passed,
        comment="" if passed else f"expected {expected!r}, got {actual!r}",
    )


def make_contains(substring: str) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff `substring` appears in the output."""
    def grader(example: Example, output: Any) -> Score:
        passed = substring in str(output)
        return Score(
            key=f"contains({substring!r})",
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"{substring!r} not found in output",
        )
    return grader


def make_regex(pattern: str) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff `pattern` matches anywhere in the output."""
    compiled = re.compile(pattern)

    def grader(example: Example, output: Any) -> Score:
        passed = bool(compiled.search(str(output)))
        return Score(
            key=f"regex({pattern!r})",
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"pattern {pattern!r} did not match output",
        )
    return grader


def make_schema_grader(model: type[BaseModel]) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff the output parses as `model` (JSON or dict)."""
    def grader(example: Example, output: Any) -> Score:
        try:
            data = json.loads(output) if isinstance(output, str) else output
            model.model_validate(data)
            return Score(key="schema", score=1.0, passed=True)
        except (ValidationError, json.JSONDecodeError, Exception) as exc:
            return Score(key="schema", score=0.0, passed=False, comment=str(exc))
    return grader


def make_golden_grader(golden: str) -> Callable[[Example, Any], Score]:
    """Return a grader that passes iff output matches a stored golden string."""
    def grader(example: Example, output: Any) -> Score:
        passed = str(output).strip() == golden.strip()
        return Score(
            key="golden",
            score=1.0 if passed else 0.0,
            passed=passed,
            comment="" if passed else f"golden mismatch: expected {golden!r}",
        )
    return grader


print("Harness loaded — Score, Example, ExampleResult, EvalReport, run_eval, "
      "exact_match, make_contains, make_regex, make_schema_grader, make_golden_grader")